# 37 · Custom model adapters and offline execution

ADK accepts BaseLlm implementations as well as Gemini. A deterministic adapter below returns fixture facts without an API request; it is a teaching stub, not a language model.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Zero model calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
import re
from google.adk.models.base_llm import BaseLlm, LlmCapabilities
from google.adk.models.llm_response import LlmResponse
from google.adk.agents import Agent

class FixtureModel(BaseLlm):
    model: str = "offline_ecommerce_fixture"

    @property
    def capabilities(self):
        return LlmCapabilities(output_schema_and_tools=False)

    async def generate_content_async(self, llm_request, stream=False):
        text = " ".join(p.text or "" for c in llm_request.contents for p in c.parts or [])
        ids = re.findall(r"O[0-9]{4}", text)
        order = get_order(ids[-1]) if ids else {"error": "Include an order ID"}
        answer = order.get("error") or f"{order['order_id']} is {order['status']}; total {order['currency']} {order['total']}."
        yield LlmResponse(content=types.Content(role="model", parts=[types.Part(text=answer)]))

agent = Agent(name="offline_shop", model=FixtureModel(), instruction="Answer from the fixture.")
lab = await make_lab(agent)
await ask(lab, "What is the status of O1001?")


Use Gemini for the live course. Other providers can be integrated through supported adapters such as LiteLLM, but need their own endpoint, credentials and optional dependencies. A local Ollama model is an alternative after installation. Keep provider selection explicit; do not silently fall back from a failed Lite call to a costly model.


## Try it

Add a second fixture response for products. To implement a real provider adapter, translate requests, tool calls, responses, usage and streaming events instead of returning a fixed response.


## Reference

[Official ADK documentation](https://adk.dev/agents/models/). Shared configuration: `config.json`. No environment activation or login cells are needed.
